# The Interrogation Architect

### Investment Banking — Banking-AI


            ## Step 0 — Notebook Header

            **Prerequisites:** Basic Python, familiarity with text analysis concepts, and comfort reading meeting-note style documents.

            **What You Will Learn:**

            - Describe how historical notes and investment hypotheses can be reused to structure a better company-meeting agenda.
- Compare keyword heuristics against a text model for predicting likely CEO response styles.
- Construct a simple simulation that suggests how a CEO may respond based on similar past language.

            **Estimated time:** 45 minutes

            **Collection context:** Investment-banking notebooks that teach how internal expertise can be preserved as structured, reusable decision-support assets.


## Step 1 — Investment-Banking Problem Introduction

**Why this problem matters:** A strong meeting agenda turns scattered internal memory into sharper questioning, which is often where firms preserve their edge.

**Operational question:** Which topics should make the agenda, and how is management most likely to respond when pushed on them?

**Primary stakeholders:** research analysts, portfolio managers, investor-relations teams, and sector specialists

**Decision supported:** Prioritize agenda topics and prepare likely management-response patterns before a high-stakes CEO meeting.

**Comprehension check:** Before looking at the data, write one sentence explaining why past management language matters even when the formal guidance has changed.

**Domain framing note:** This notebook is educational and synthetic. It demonstrates decision support for investment-banking and adjacent institutional workflows, not production approval or investment advice.


## Step 2 — Environment Setup

Use synthetic meeting notes and response templates so the workflow stays shareable without depending on proprietary transcripts.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)
RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)

print("Environment ready for a synthetic meeting-preparation workflow.")


## Step 3 — Data Creation & Context

We simulate historic meeting notes with recurring themes and management response styles. The point is to preserve reasoning patterns, not verbatim internal content.


In [ ]:
themes = ["margin", "pricing", "capex", "hiring", "product_timing", "capital_return"]
styles = {
    "deflect": ["we are not focused on quarter-to-quarter noise", "it is too early to be specific", "we manage the business for the long term"],
    "quantify": ["the impact is roughly 120 basis points", "we see a mid-single-digit range", "inventory should normalize in two quarters"],
    "reframe": ["the more important issue is product mix", "we think the debate should center on quality", "customer retention matters more than volume"],
    "confirm": ["that is directionally correct", "your thesis is broadly aligned", "we agree the driver is demand stability"],
}
records = []

for _ in range(900):
    theme = RNG.choice(themes)
    style = RNG.choice(list(styles), p=[0.26, 0.30, 0.24, 0.20])
    prompt = f"question on {theme} and current investment hypothesis"
    response = f"{prompt} management response: {RNG.choice(styles[style])}"
    records.append((theme, response, style))

meeting_df = pd.DataFrame(records, columns=["theme", "response_text", "response_style"])
meeting_df["model_text"] = "theme_" + meeting_df["theme"] + " " + meeting_df["response_text"]
print(meeting_df.head(3).to_string(index=False))


## Step 4 — Exploratory Data Analysis

Explore response-style frequency and how different themes show up in the synthetic archive before training the model.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=meeting_df, x="response_style", color="#66C2A5", ax=axes[0])
axes[0].set_title("Historic CEO Response Styles")
axes[0].set_xlabel("Response Style")
axes[0].set_ylabel("Count")

theme_mix = meeting_df.groupby(["theme", "response_style"]).size().reset_index(name="count")
sns.barplot(data=theme_mix, x="theme", y="count", hue="response_style", ax=axes[1])
axes[1].set_title("Response Styles by Theme")
axes[1].set_xlabel("Theme")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()
plt.close()


Alt-Text: The charts show which management response styles dominate the historical archive and how those styles vary across themes such as margins, pricing, or capital return.


## Step 5 — Feature Engineering

We keep the features interpretable by prepending theme tags to the response text instead of burying all structure in a black box.


In [ ]:
analysis_df = meeting_df.copy()
analysis_df["theme_token"] = "theme_" + analysis_df["theme"]
analysis_df["retrieval_text"] = analysis_df["theme_token"] + " " + analysis_df["response_text"]

print(analysis_df[["theme", "response_style", "retrieval_text"]].head(3).to_string(index=False))


## Step 6 — Baseline Establishment

A basic meeting-prep baseline is a keyword heuristic that guesses response style from familiar management phrases.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    analysis_df["retrieval_text"],
    analysis_df["response_style"],
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=analysis_df["response_style"],
)

def keyword_style(text: str) -> str:
    if "basis points" in text or "range" in text or "quarters" in text:
        return "quantify"
    if "directionally correct" in text or "agree" in text:
        return "confirm"
    if "more important" in text or "debate" in text:
        return "reframe"
    return "deflect"

baseline_pred = X_test.apply(keyword_style)
print(f"Baseline accuracy: {accuracy_score(y_test, baseline_pred):.3f}")


## Step 7 — Model Training & Evaluation

Train a text model that predicts likely response style from historic language. Prediction prompt: which style will be easiest to separate?


In [ ]:
response_model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("classifier", LogisticRegression(max_iter=1200)),
    ]
)
response_model.fit(X_train, y_train)
model_pred = response_model.predict(X_test)

print(f"Model accuracy: {accuracy_score(y_test, model_pred):.3f}")
print(classification_report(y_test, model_pred))


## Step 8 — Interpretability & Explainability

Top-weighted terms help users judge whether the model is recognizing real management patterns or just memorizing trivial phrases.


In [ ]:
vectorizer = response_model.named_steps["tfidf"]
classifier = response_model.named_steps["classifier"]
feature_names = vectorizer.get_feature_names_out()

for class_index, class_name in enumerate(classifier.classes_):
    top_terms = feature_names[np.argsort(classifier.coef_[class_index])[-8:]][::-1]
    print(f"Top signals for {class_name}: {', '.join(top_terms)}")


## Step 9 — Operational Application

Turn the archive into a practical agenda builder by retrieving similar historical responses for a set of current hypotheses.


In [ ]:
current_agenda = pd.DataFrame(
    {
        "agenda_item": [
            "Ask about gross margin durability after pricing changes",
            "Push on the capex timing for the new platform",
            "Test whether buybacks remain the preferred use of cash",
        ]
    }
)

tfidf = response_model.named_steps["tfidf"]
agenda_matrix = tfidf.transform(current_agenda["agenda_item"])
archive_matrix = tfidf.transform(analysis_df["retrieval_text"])
similarity = cosine_similarity(agenda_matrix, archive_matrix)

for row_index, agenda_item in enumerate(current_agenda["agenda_item"]):
    match_index = similarity[row_index].argmax()
    matched_row = analysis_df.iloc[match_index]
    predicted_style = response_model.predict([agenda_item])[0]
    print(f"Agenda item: {agenda_item}")
    print(f"Likely response style: {predicted_style}")
    print(f"Closest archive note: {matched_row['response_text']}")
    print("---")


            ## Step 10 — Summary & Reflection

            **What you accomplished:** You built a synthetic meeting-preparation workflow that learns from historical management language to rank agenda items and anticipate likely responses.

            **Limitations to note:**

            - The note archive is synthetic and lacks the nuance of real executive communication.
- Response-style labels are simplified and do not capture mixed or deliberately ambiguous answers.
- A text model can support preparation, but it should never replace analyst judgment in live meetings.

            **Ethical reflection:** Preserving institutional memory can improve questioning, but teams must avoid turning historical language patterns into rigid assumptions that silence new information.

            **Reflection questions:**

            1. Which additional internal artifact would make the agenda builder more useful: previous notes, earnings calls, or model-change logs?
2. How would you evaluate whether a simulated response is directionally helpful rather than merely fluent?
3. What privacy or governance controls should exist around an internal meeting archive?


            ## References

            - Manning, C., Raghavan, P., & Schutze, H. Introduction to Information Retrieval.
- Scikit-learn user guide: text classification and cosine similarity workflows.
- Scenario inspiration: buy-side and sell-side meeting preparation practices using historical note archives.
